# Colab Setup - Clone from Git and Prepare Data

Run this notebook first in Colab to set up the environment.

# Clone your repository
REPO_URL = "https://github.com/Echo-Lee/ORIE-5981-RAG.git"

# Try cloning
import os
import subprocess

try:
    # First check if already cloned
    if os.path.exists('ORIE-5981-RAG'):
        print("✅ Repository already exists, pulling latest changes...")
        os.chdir('ORIE-5981-RAG')
        !git pull
    else:
        print("Cloning repository...")
        result = subprocess.run(['git', 'clone', REPO_URL], capture_output=True, text=True)
        
        if result.returncode != 0:
            print("⚠️ Clone failed with authentication. Trying alternative method...")
            # Alternative: Download as zip
            import urllib.request
            import zipfile
            
            zip_url = "https://github.com/Echo-Lee/ORIE-5981-RAG/archive/refs/heads/main.zip"
            print(f"Downloading from {zip_url}")
            
            urllib.request.urlretrieve(zip_url, "repo.zip")
            with zipfile.ZipFile("repo.zip", 'r') as zip_ref:
                zip_ref.extractall(".")
            os.rename("ORIE-5981-RAG-main", "ORIE-5981-RAG")
            os.remove("repo.zip")
            print("✅ Downloaded and extracted from zip")
        
        os.chdir('ORIE-5981-RAG')
    
    print(f"✅ Current directory: {os.getcwd()}")
    !ls -la

except Exception as e:
    print(f"❌ Error: {e}")
    print("\n💡 Manual solution:")
    print("1. Make sure your GitHub repo is PUBLIC")
    print("2. Or download the code manually and upload to Colab")
    print("3. Or use a GitHub token for authentication")

In [ ]:
# Clone your repository
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # CHANGE THIS

!git clone {REPO_URL}

# Extract repo name from URL
import os
repo_name = REPO_URL.split("/")[-1].replace(".git", "")
os.chdir(repo_name)

print(f"Current directory: {os.getcwd()}")
!ls -la

## 2. Mount Google Drive and Link Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create data directories
!mkdir -p data/processed/hospital
!mkdir -p data/processed/corruption

# Option A: Link to Google Drive
# Upload your data to: Google Drive/Capstone-Data/
DRIVE_DATA_PATH = "/content/drive/MyDrive/Capstone-Data"  # CHANGE THIS

# Create symlinks
!ln -sf {DRIVE_DATA_PATH}/hospital/threads_with_summary.json data/processed/hospital/
!ln -sf {DRIVE_DATA_PATH}/corruption/emails_group_by_thread.json data/processed/corruption/

# Verify data files
print("\nData files:")
!ls -lh data/processed/hospital/
!ls -lh data/processed/corruption/

## 3. Install Dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu gradio pyyaml openai tqdm

# Check GPU
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 4. Setup API Keys

In [ ]:
# Option A: Use Colab Secrets (Recommended)
# Go to: 🔑 Secrets (left sidebar) → Add new secret
# Add: AZURE_API_KEY and AZURE_ENDPOINT

from google.colab import userdata
import yaml

try:
    api_key = userdata.get('AZURE_API_KEY')
    endpoint = userdata.get('AZURE_ENDPOINT')
    print("✅ Retrieved API keys from Colab Secrets")
except:
    print("⚠️  Colab Secrets not found. Using manual input...")
    api_key = input("Enter Azure API Key: ")
    endpoint = input("Enter Azure Endpoint: ")

# Create config files from templates
for dataset in ['hospital', 'corruption']:
    template_path = f'experiments/{dataset}_base_template.yaml'
    config_path = f'experiments/{dataset}_base.yaml'
    
    with open(template_path, 'r') as f:
        config = yaml.safe_load(f)
    
    config['azure_api_key'] = api_key
    config['azure_endpoint'] = endpoint
    
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print(f"✅ Created {config_path}")

print("\n✅ Setup complete! You can now run launcher.ipynb")

## 5. Verify Setup

In [ ]:
# Check everything is ready
import os
from pathlib import Path

checks = [
    ("Hospital data", Path("data/processed/hospital/threads_with_summary.json").exists()),
    ("Corruption data", Path("data/processed/corruption/emails_group_by_thread.json").exists()),
    ("Hospital config", Path("experiments/hospital_base.yaml").exists()),
    ("Corruption config", Path("experiments/corruption_base.yaml").exists()),
    ("Source code", Path("src/config/config.py").exists()),
]

print("Setup Verification:")
print("=" * 50)
all_good = True
for name, status in checks:
    icon = "✅" if status else "❌"
    print(f"{icon} {name}")
    if not status:
        all_good = False

print("=" * 50)
if all_good:
    print("\n🎉 All checks passed! Ready to run launcher.ipynb")
else:
    print("\n⚠️  Some checks failed. Please fix before proceeding.")

## Next Steps

1. ✅ This setup is complete
2. Open `notebooks/launcher.ipynb`
3. Set MODE and DATASET
4. Run all cells to build index and launch demo